# Scenario: Hunting for "Quantum" Clinicians


In [1]:
import pandas as pd
import sqlite3
# creating dataset of scheduled surgical blocks with overlapping hour
schedule_data = {
    "event_id": [501, 502, 503, 504, 505],
    "provider_name": ["Dr. Lin", "Dr. Lin", "Dr. Patel", "Dr. Patel", "Dr. Lin"],
    "location": ["OR-1", "OR-3", "Clinic-A", "OR-2", "OR-1"],
    "start_time": ["10:00", "10:30", "09:00", "13:00", "14:00"],
    "end_time":   ["11:30", "12:00", "12:00", "15:00", "16:00"]
    # Look closely at event 501 and 502 for Dr. Lin! 10:30 starts before 11:30 ends!
}
# adding dataset to DataFrame
df_schedule_data = pd.DataFrame(schedule_data)
# creating sql connection and save the data into temp memory
connt = sqlite3.connect(":memory:")
df_schedule_data.to_sql("provider_schedules", connt, index = False, if_exists = "replace")
# function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("************************** Concurrency Audit Database is ready! ***************")

************************** Concurrency Audit Database is ready! ***************


# Finding the Overlapping Events

In [5]:
# all data to review
all_data = "SELECT * FROM provider_schedules"
print("************************************** all data to review ***************")
display(run_query(all_data))
print()
# SQL query that performs a Self-Join on the provider_schedules table to find rows where the same doctor is scheduled for two different locations at the same time.
overlapping_events = """
SELECT a.provider_name,
       a.event_id AS event_A, a.location AS loc_A, a.start_time AS start_A, a.end_time AS end_A,
       b.event_id AS event_B, b.location AS loc_B, b.start_time AS start_B, b.end_time AS end_B
FROM provider_schedules a
INNER JOIN provider_schedules b 
   ON a.provider_name = b.provider_name 
  AND a.event_id < b.event_id -- Prevents showing duplicates of the same pair
WHERE a.start_time < b.end_time 
  AND b.start_time < a.end_time;
"""
print("********************************* Overlapping Events ************************")
display(run_query(overlapping_events))

************************************** all data to review ***************


,event_id,provider_name,location,start_time,end_time
0,501,Dr. Lin,OR-1,10:00,11:30
1,502,Dr. Lin,OR-3,10:30,12:00
2,503,Dr. Patel,Clinic-A,09:00,12:00
3,504,Dr. Patel,OR-2,13:00,15:00
4,505,Dr. Lin,OR-1,14:00,16:00



********************************* Overlapping Events ************************


,provider_name,event_A,loc_A,start_A,end_A,event_B,loc_B,start_B,end_B
0,Dr. Lin,501,OR-1,10:00,11:30,502,OR-3,10:30,12:00
